# Mechanistic Interpretability of a Hyperspectral Vision Transformer
### Real PyTorch version — run in Colab (GPU optional; this model is small)

This is the PyTorch counterpart to the NumPy-autograd version validated in
this project's sandbox (`src/vit_model.py` + `src/interpretability.py`,
built on a from-scratch, gradient-checked autograd engine because this
sandbox has no internet access to install torch). Same architecture, same
three interpretability techniques (logit lens, attention rollout, gradient
x input band attribution) — here using real `torch.autograd`, real
`register_forward_hook` activation extraction, and `nn.MultiheadAttention`
with `need_weights=True` for attention maps.

**Compute reality check:** ~160k parameters, tiny 16x16x135 cubes. Trains
in well under a minute on Colab's free CPU tier. No GPU required at this
scale — only matters once you're training on real multi-thousand-pixel
Pixxel Firefly scenes with a much deeper/wider model.

In [ ]:
!pip install -q torch numpy matplotlib
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Data — upload `hyperspectral_cube_simulator.py` (and the earlier project's `spectral_physics_simulator.py` for full physical realism) to Colab, or paste contents

In [ ]:
import sys; sys.path.append('/content')
from hyperspectral_cube_simulator import simulate_vit_dataset, simulate_scene, SCENE_TYPES, FIREFLY_WAVELENGTHS

cubes, labels, grids, wl = simulate_vit_dataset(n_per_class=60, H=16, W=16, seed=0)
flat = cubes.reshape(-1, cubes.shape[-1])
band_mean, band_std = flat.mean(0), flat.std(0) + 1e-6
cubes_norm = (cubes - band_mean) / band_std
print(cubes.shape, SCENE_TYPES)

## 2. Real PyTorch Vision Transformer, with attention weights exposed for interpretability

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d, n_heads, mlp_hidden):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, mlp_hidden), nn.GELU(), nn.Linear(mlp_hidden, d))
        self.last_attn = None

    def forward(self, x):
        h = self.ln1(x)
        attn_out, attn_w = self.attn(h, h, h, need_weights=True, average_attn_weights=False)
        self.last_attn = attn_w.detach()  # (batch, heads, seq, seq)
        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x

class HyperspectralViT(nn.Module):
    def __init__(self, n_bands, patch_size, grid_size, n_classes, embed_dim=48, n_heads=4, n_layers=3, mlp_hidden=96):
        super().__init__()
        self.patch_size, self.grid_size = patch_size, grid_size
        patch_dim = patch_size * patch_size * n_bands
        n_patches = grid_size * grid_size
        self.patch_embed = nn.Linear(patch_dim, embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, embed_dim) * 0.02)
        self.blocks = nn.ModuleList([EncoderBlock(embed_dim, n_heads, mlp_hidden) for _ in range(n_layers)])
        self.ln_final = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, n_classes)
        self.layer_outputs = []  # populated by forward hooks below

    def patchify(self, cube):  # cube: (batch, H, W, bands)
        b, H, W, B = cube.shape
        ps, gs = self.patch_size, self.grid_size
        cube = cube.reshape(b, gs, ps, gs, ps, B).permute(0, 1, 3, 2, 4, 5)
        return cube.reshape(b, gs * gs, ps * ps * B)

    def forward(self, cube, capture=False):
        patches = self.patchify(cube)
        emb = self.patch_embed(patches)
        cls = self.cls_token.expand(cube.shape[0], -1, -1)
        x = torch.cat([cls, emb], dim=1) + self.pos_embed
        self.layer_outputs = []
        for block in self.blocks:
            x = block(x)
            if capture:
                self.layer_outputs.append(x.detach().clone())
        x = self.ln_final(x)
        logits = self.head(x[:, 0, :])
        return logits

## 3. Train (real backprop via torch.autograd)

In [ ]:
Xt = torch.tensor(cubes_norm, dtype=torch.float32, device=device)
yt = torch.tensor(labels, dtype=torch.long, device=device)
n = len(yt); perm = torch.randperm(n)
split = int(0.8*n)
train_idx, val_idx = perm[:split], perm[split:]

model = HyperspectralViT(n_bands=135, patch_size=4, grid_size=4, n_classes=4).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)

for epoch in range(30):
    model.train()
    perm_ep = train_idx[torch.randperm(len(train_idx))]
    total_loss = 0
    for i in range(0, len(perm_ep), 16):
        idx = perm_ep[i:i+16]
        logits = model(Xt[idx])
        loss = F.cross_entropy(logits, yt[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item() * len(idx)
    model.eval()
    with torch.no_grad():
        val_acc = (model(Xt[val_idx]).argmax(-1) == yt[val_idx]).float().mean().item()
    if epoch % 5 == 0 or epoch == 29:
        print(f'epoch {epoch:2d} loss={total_loss/len(train_idx):.4f} val_acc={val_acc:.3f}')

## 4. Logit Lens — apply the trained head to every intermediate layer's CLS token

In [ ]:
def logit_lens_torch(model, cube_norm_single):
    model.eval()
    with torch.no_grad():
        x = cube_norm_single.unsqueeze(0)
        _ = model(x, capture=True)
        rows = []
        for layer_out in model.layer_outputs:
            cls_raw = layer_out[:, 0, :]
            normed = model.ln_final(cls_raw)
            logits = model.head(normed)
            rows.append(F.softmax(logits, dim=-1)[0].cpu().numpy())
    return np.array(rows)

test_cube, _, _ = simulate_scene('early_blight_patch', H=16, W=16, seed=9999)
test_norm = torch.tensor((test_cube - band_mean) / band_std, dtype=torch.float32, device=device)
print('Per-layer class probabilities:')
print(logit_lens_torch(model, test_norm))

## 5. Attention Rollout — real attention weights from `nn.MultiheadAttention`

In [ ]:
def attention_rollout_torch(model, cube_norm_single):
    model.eval()
    with torch.no_grad():
        x = cube_norm_single.unsqueeze(0)
        _ = model(x, capture=True)
        seq_len = model.blocks[0].last_attn.shape[-1]
        rollout = torch.eye(seq_len)
        for block in model.blocks:
            avg_heads = block.last_attn[0].mean(0)  # (seq, seq), average over heads
            avg_heads = avg_heads + torch.eye(seq_len)
            avg_heads = avg_heads / avg_heads.sum(-1, keepdim=True)
            rollout = avg_heads @ rollout
        cls_relevance = rollout[0, 1:]
        return (cls_relevance / cls_relevance.sum()).cpu().numpy()

rollout = attention_rollout_torch(model, test_norm)
print('Attention rollout (4x4 patch grid):')
print(rollout.reshape(4, 4).round(3))

## 6. Spectral Band Attribution — gradient x input via real `torch.autograd.grad`

In [ ]:
def band_attribution_torch(model, cube_norm_single, target_class, band_std):
    model.eval()
    x = cube_norm_single.unsqueeze(0).clone().requires_grad_(True)
    logits = model(x)
    target_logit = logits[0, target_class]
    grad, = torch.autograd.grad(target_logit, x)
    grad_x_input = (grad * x).detach().cpu().numpy()[0]     # (H, W, bands)
    band_scores = np.abs(grad_x_input).mean(axis=(0, 1)) / (band_std + 1e-8)
    return band_scores

scores = band_attribution_torch(model, test_norm, SCENE_TYPES.index('early_blight_patch'), band_std)
top10 = np.argsort(-scores)[:10]
for i in top10:
    print(f'{FIREFLY_WAVELENGTHS[i]:.0f}nm : importance={scores[i]:.4f}')

## Next steps to make this real
1. Swap the simulator for a real Pixxel Firefly L2A cube (same `(H,W,bands)` shape contract).
2. Scale to real scene sizes (hundreds x hundreds of pixels) and re-tune patch size / grid size accordingly — attention rollout and gradient x input both scale linearly, no architecture change needed.
3. Cross-validate band attribution against the earlier Sparse Autoencoder project's monosemantic feature dashboard — if both independently point to the same wavelengths for the same phenomenon, that's strong evidence the signal is physical, not an artifact of either architecture.
4. Run attention rollout and logit lens across MANY scenes of the same disease class and check consistency — a single-scene result (as shown here) is a demo, not a validated finding.